# 🎲 Music Generation — Markov Chain Baseline
**CSE425 Neural Networks | Ummay Maimona Chaman | 22301719 | Section 1**

This notebook implements a simple **Markov Chain** baseline for music generation. It learns transition probabilities between notes from the real dataset and generates new sequences for comparison with the neural-based models (AE, VAE, Transformer).

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

from src.config import TRAIN_TEST_DIR, MIDI_OUT_DIR
from src.generation.midi_export import piano_roll_to_midi

plt.style.use('seaborn-v0_8-bright')
print("Setup complete!")

## 1. Load Real Data
We use the same preprocessed data used for training the neural models.

In [ ]:
X = np.load(os.path.join(TRAIN_TEST_DIR, 'ae_train.npy'))
print(f"Training on {X.shape[0]} segments of shapes {X.shape[1:]}")

## 2. Markov Logic
We learn $P(note_{t} | note_{t-1})$.

In [ ]:
def build_markov_model(data):
    transitions = defaultdict(lambda: defaultdict(int))
    for seq in data[:500]:
        # Get the first active pitch in each timestep to simplify
        active_notes = []
        for t in range(seq.shape[0]):
            pitches = np.where(seq[t] > 0.5)[0]
            if len(pitches) > 0:
                active_notes.append(pitches[0])
            else:
                active_notes.append(-1) # Rest
        
        for i in range(len(active_notes) - 1):
            transitions[active_notes[i]][active_notes[i+1]] += 1
            
    # Normalize probabilities
    prob_map = {}
    for state, counts in transitions.items():
        total = sum(counts.values())
        prob_map[state] = {k: v/total for k, v in counts.items()}
    return prob_map

model = build_markov_model(X)
print(f"Markov model built with {len(model)} states.")

## 3. Generate Sequences

In [ ]:
def generate_markov(model, length=64):
    current = np.random.choice(list(model.keys()))
    result = [current]
    for _ in range(length - 1):
        if current in model:
            next_choices = list(model[current].keys())
            next_probs = list(model[current].values())
            current = np.random.choice(next_choices, p=next_probs)
        else:
            current = np.random.choice(list(model.keys()))
        result.append(current)
    
    # Convert pitches back to piano roll
    roll = np.zeros((length, X.shape[2]))
    for t, p in enumerate(result):
        if p != -1: roll[t, p] = 1.0
    return roll

gen_roll = generate_markov(model)
plt.figure(figsize=(10, 3))
plt.imshow(gen_roll.T, aspect='auto', origin='lower', cmap='Blues')
plt.title("Markov-Generated Piano Roll")
plt.xlabel("Time Step")
plt.ylabel("Pitch Bin")
plt.show()

## 4. Export as MIDI

In [ ]:
out_path = os.path.join(MIDI_OUT_DIR, 'markov_baseline_sample.mid')
piano_roll_to_midi(gen_roll.T, fs=4.0, path=out_path)
print(f"Saved Markov baseline to {out_path}")